# Using Kerchunk and Icechunk with grib2io and Xarray

This notebook demonstrates how to virtualize GRIB2 files using Kerchunk and store them in a cloud-native, versioned format using Icechunk.

## Notebook Convention

Use the standard grib2io xarray backend interface directly.
Pass grib2io-specific options as keyword arguments to `xr.open_dataset(...)` or `xr.open_mfdataset(...)` (for example `use_icechunk`, `storage_options`, `filters`, `max_workers`, `network_timeout`, and `max_concurrent_requests`).

In [ ]:
import grib2io
import xarray as xr
import pathlib
import os
import json
from grib2io.kerchunk import ReferenceGenerator

## 1. Generate Kerchunk References
We use `ReferenceGenerator` to scan a GRIB2 file and create a manifest of byte-range references.

In [ ]:
# Use the GFS test file bundled with grib2io.
# Assumes the notebook is run from the project root or demos/ directory.
_here = pathlib.Path(os.path.abspath(""))
_candidates = [
    _here / "tests" / "input_data" / "gfs.t00z.pgrb2.1p00.f024",
    _here.parent / "tests" / "input_data" / "gfs.t00z.pgrb2.1p00.f024",
]
grib2_file = next(str(p) for p in _candidates if p.exists())

gen = ReferenceGenerator(grib2_file)
manifest = gen.generate()
print(f"Generated manifest with {len(manifest['refs'])} references")

## 2. Open as an xarray Dataset

Use the standard grib2io/xarray interface by opening the generated manifest with
`engine="grib2io"`. This keeps dataset access routed through the existing UI.

Notebook convention: pass grib2io-specific options directly as keyword arguments
to `xr.open_dataset(...)` / `xr.open_mfdataset(...)` (rather than nesting them
under a separate backend args dict).

For an even shorter path when you have the file URL directly, use:
`xr.open_dataset(url, engine="grib2io", use_icechunk=True)`


In [ ]:
manifest_path = pathlib.Path(grib2_file).with_suffix(".kerchunk.json")
manifest_path.write_text(json.dumps(manifest))

ds = xr.open_dataset(manifest_path, engine="grib2io")
display(ds)

## 4. Efficient Variable Streaming
Because the data is lazily indexed, we can "stream" or access specific parts of a variable without loading the entire dataset into memory. Using `chunks={}` in `open_dataset` ensures that Xarray uses Dask for lazy loading.

This is particularly powerful for large variables like temperature (TMP) or humidity (RH) across multiple vertical levels or time steps.

In [ ]:
if "TMP" in ds.data_vars:
    # Select a spatial subset — only triggers byte-range requests for the relevant GRIB messages.
    tmp_subset = ds.TMP.isel(y=slice(0, 100), x=slice(0, 100))

    # Trigger a compute to see the data
    data = tmp_subset.compute()
    print("Successfully streamed TMP subset data.")
    display(data)

## 5. Multiple Variable Streaming
We can also stream multiple variables simultaneously. Dask will manage the parallel fetch of these chunks from the underlying storage (e.g. S3 or local disk) via the Icechunk references.

In [ ]:
# Select multiple variables for a specific region
available = [v for v in ["TMP", "HGT"] if v in ds.data_vars]
if available:
    subset_ds = ds[available].isel(y=slice(50, 150), x=slice(50, 150))

    # Dask manages parallel byte-range fetches from the Icechunk store.
    computed_result = subset_ds.compute()
    print(f"Streamed {available} successfully.")
    display(computed_result)